# Renombrado físico: reflejar en el archivo la especie ya confirmada

Cuando una foto pasa a `estado='usable'` (confirmada, con `especie_id`), la base de datos ya lo sabe pero el archivo en `Y:\Galería\Fotos_Unificadas\` todavía trae el nombre de cuando se migró — que puede ser `SINESP`, un nombre común (`Aguacate`), o un código de espécimen. Este notebook actualiza el segmento de especie en el nombre físico para que quede el nombre científico confirmado, manteniendo intactos rama, autor y hash (el hash sale de `ruta_original`, no cambia aunque el archivo se renombre).

**Regla de oro:** siempre se lee y se escribe sobre `fotos.ruta_actual`, nunca sobre `staging_escaneo.ruta_completa` (esa es historia, no la ubicación real).

**Patrón de seguridad del proyecto:** primero se arma un plan en seco (`Plan_Renombrado_Fisico.xlsx`), se revisa, y solo después se prende `EJECUTAR_RENOMBRADO_REAL = True`. Es resumible: si se corta a la mitad, correrlo de nuevo solo procesa lo que falte (lo ya renombrado ya no aparece como pendiente, porque `ruta_actual` en la base ya quedó actualizada).

In [1]:
import json
import re
import sqlite3
from datetime import datetime
from pathlib import Path, PureWindowsPath

import pandas as pd

def cargar_config(archivo="config_local.json"):
    ruta = Path(archivo)
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontro {archivo}. Copia config_local.example.json y ajustalo.")
    return json.loads(ruta.read_text(encoding="utf-8"))

config = cargar_config()
RUTA_BD = Path("../../experimentos/bd/campuslab_volumen.db")

con = sqlite3.connect(RUTA_BD)
con.execute("""
    CREATE TABLE IF NOT EXISTS renombrados_fisicos (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        foto_id INTEGER,
        ruta_anterior TEXT,
        ruta_nueva TEXT,
        fecha TEXT
    )
""")
con.commit()
print(f"Base de datos: {RUTA_BD.resolve()} ({'existe' if RUTA_BD.exists() else 'NO ENCONTRADA'})")

Base de datos: C:\Users\hecto\OneDrive\Escritorio\ITESO\Comunidad\Campus Lab\experimentos\bd\campuslab_volumen.db (existe)


## 1. Armar el plan (no toca nada todavía)

In [2]:
# Mismo slug() que uso Migracion_Volumen.ipynb al nombrar los archivos por primera vez,
# para que un archivo ya bien nombrado nunca aparezca como "cambia" por una diferencia de criterio.
def slug(texto):
    if not texto:
        return None
    s = re.sub(r"[^\w\s-]", "", texto, flags=re.UNICODE)
    s = re.sub(r"\s+", "", s)
    return s or None

PATRON_NOMBRE = re.compile(r"^(AUTORES|ESPECIE|EVENTOS|ESPACIOS)-(.+)-([0-9a-f]{6})(\.[^.]+)$", re.IGNORECASE)

def calcular_nuevo_nombre(nombre_archivo, nombre_cientifico):
    """Devuelve (nombre_nuevo, motivo_si_no_se_pudo). Solo cambia el segmento de especie."""
    m = PATRON_NOMBRE.match(nombre_archivo)
    if not m:
        return None, "nombre_no_coincide_con_convencion"
    rama, medio, hash6, ext = m.groups()
    partes = medio.split("-", 1)
    autor_parte = partes[0]
    especie_parte_nueva = slug(nombre_cientifico)
    if not especie_parte_nueva:
        return None, "nombre_cientifico_vacio_o_no_sluggeable"
    return f"{rama}-{autor_parte}-{especie_parte_nueva}-{hash6}{ext}", None


df = pd.read_sql(
    "SELECT f.id AS foto_id, f.ruta_actual, e.nombre_cientifico "
    "FROM fotos f JOIN especies e ON e.id = f.especie_id "
    "WHERE f.estado = 'usable' ORDER BY f.id",
    con,
)

def separar(ruta_str):
    p = PureWindowsPath(ruta_str)
    return str(p.parent), p.name

df[["carpeta", "nombre_actual"]] = df["ruta_actual"].apply(lambda r: pd.Series(separar(r)))

resultado = df.apply(lambda fila: calcular_nuevo_nombre(fila["nombre_actual"], fila["nombre_cientifico"]), axis=1)
df["nombre_nuevo"] = resultado.apply(lambda t: t[0])
df["motivo_sin_cambio"] = resultado.apply(lambda t: t[1])
df["ruta_nueva"] = df.apply(
    lambda fila: str(PureWindowsPath(fila["carpeta"]) / fila["nombre_nuevo"]) if fila["nombre_nuevo"] else None,
    axis=1,
)
df["cambia"] = df["nombre_nuevo"].notna() & (df["nombre_nuevo"] != df["nombre_actual"])

print(f"Fotos usable con especie: {len(df)}")
print(f"Ya bien nombradas (no cambian): {(~df['cambia'] & df['nombre_nuevo'].notna()).sum()}")
print(f"Se renombrarían:                {df['cambia'].sum()}")
print(f"No se pudieron calcular:        {df['nombre_nuevo'].isna().sum()}")
if df["nombre_nuevo"].isna().any():
    print(df[df["nombre_nuevo"].isna()]["motivo_sin_cambio"].value_counts())

Fotos usable con especie: 943
Ya bien nombradas (no cambian): 0
Se renombrarían:                943
No se pudieron calcular:        0


In [3]:
CARPETA_SALIDA = Path("../salida")
CARPETA_SALIDA.mkdir(exist_ok=True)

plan = df[df["cambia"]][["foto_id", "carpeta", "nombre_actual", "nombre_nuevo", "nombre_cientifico"]]
plan.to_excel(CARPETA_SALIDA / "Plan_Renombrado_Fisico.xlsx", index=False)
print(f"Plan guardado: {CARPETA_SALIDA / 'Plan_Renombrado_Fisico.xlsx'} ({len(plan)} filas)")
print("\nEjemplos:")
print(plan.head(10).to_string(index=False))

Plan guardado: ..\salida\Plan_Renombrado_Fisico.xlsx (943 filas)

Ejemplos:
 foto_id                     carpeta                        nombre_actual                                nombre_nuevo nombre_cientifico
    1207 Y:\Galería\Fotos_Unificadas ESPECIE-SINAUTOR-Aguacate-84e8af.jpg ESPECIE-SINAUTOR-Perseaamericana-84e8af.jpg  Persea americana
    1208 Y:\Galería\Fotos_Unificadas ESPECIE-SINAUTOR-Aguacate-8eab34.jpg ESPECIE-SINAUTOR-Perseaamericana-8eab34.jpg  Persea americana
    1209 Y:\Galería\Fotos_Unificadas ESPECIE-SINAUTOR-Aguacate-450ddc.jpg ESPECIE-SINAUTOR-Perseaamericana-450ddc.jpg  Persea americana
    1210 Y:\Galería\Fotos_Unificadas ESPECIE-SINAUTOR-Aguacate-761872.jpg ESPECIE-SINAUTOR-Perseaamericana-761872.jpg  Persea americana
    1211 Y:\Galería\Fotos_Unificadas ESPECIE-SINAUTOR-Aguacate-a9c3a8.jpg ESPECIE-SINAUTOR-Perseaamericana-a9c3a8.jpg  Persea americana
    1212 Y:\Galería\Fotos_Unificadas ESPECIE-SINAUTOR-Aguacate-ac16c2.jpg ESPECIE-SINAUTOR-Perseaamericana-a

## 2. Aplicar de verdad

Revisa `Plan_Renombrado_Fisico.xlsx` primero. Solo entonces cambia la bandera de abajo a `True`.

Requiere acceso a `Y:\Galería` (red de ITESO).

In [4]:
EJECUTAR_RENOMBRADO_REAL = True  # <-- cambialo a True despues de revisar el plan en seco

if not EJECUTAR_RENOMBRADO_REAL:
    print("EJECUTAR_RENOMBRADO_REAL esta en False. No se renombro nada todavia.")
else:
    renombrados = 0
    saltados_ya_existe_destino = 0
    errores = 0
    for _, fila in df[df["cambia"]].iterrows():
        ruta_vieja = Path(fila["ruta_actual"])
        ruta_nueva = Path(fila["ruta_nueva"])
        try:
            if not ruta_vieja.exists():
                print(f"  [falta el original] {ruta_vieja}")
                errores += 1
                continue
            if ruta_nueva.exists():
                print(f"  [ya existe el destino, no se toca] {ruta_nueva}")
                saltados_ya_existe_destino += 1
                continue
            ruta_vieja.rename(ruta_nueva)
            con.execute("UPDATE fotos SET ruta_actual=? WHERE id=?", (str(ruta_nueva), int(fila["foto_id"])))
            con.execute(
                "INSERT INTO renombrados_fisicos (foto_id, ruta_anterior, ruta_nueva, fecha) VALUES (?,?,?,?)",
                (int(fila["foto_id"]), str(ruta_vieja), str(ruta_nueva), datetime.now().isoformat(timespec="seconds")),
            )
            renombrados += 1
            if renombrados % 100 == 0:
                con.commit()
                print(f"  {renombrados} renombrados...")
        except OSError as e:
            print(f"  [error] {ruta_vieja} -> {ruta_nueva}: {e}")
            errores += 1
    con.commit()
    print(f"\nListo. renombrados={renombrados}, saltados(destino ya existia)={saltados_ya_existe_destino}, errores={errores}")

  100 renombrados...


  200 renombrados...


  300 renombrados...


  400 renombrados...


  500 renombrados...


  600 renombrados...


  700 renombrados...


  800 renombrados...


  900 renombrados...



Listo. renombrados=943, saltados(destino ya existia)=0, errores=0
